# Parkinson AI — WideResNet + Focal Loss → DeepRBF → Combined inference

This notebook runs the three stages in order:

1. Train fold-specific **WideResNet50-2 with Focal Loss** and save the baseline checkpoints.
2. Freeze each trained WideResNet, extract 2048-D pooled features, and train the **class-specific affine DeepRBF** head independently.
3. Select fold-local rejection rules where validation supports them, use a pooled out-of-fold fallback for empty/infeasible folds, export combined inference bundles, and run validation-only predictions.

**Important:** run a pilot on fold 1 first. Only change `FOLD_END` to 25 after the pilot completes and the outputs look correct. The notebook does not load the held-out test partition.

## 0. Environment and data paths

Run this notebook after applying `apply_refactor.py` from the accompanying refactor package. The project root must contain `train.py`, `train_deep_rbf.py`, and the updated DeepRBF modules.

`DATA_ROOT` must be the directory containing `datasets/annotations/fold_manifest.csv` and the corresponding `datasets/raw/` tree. If your dataset is stored elsewhere, set `PARKINSON_DATA_ROOT` or edit the candidate paths below.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import json

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/ParkinsonAiResearch")
    if not (PROJECT_DIR / "train.py").is_file():
        subprocess.run(
            ["git", "clone", "https://github.com/Aihan-2005/ParkinsonAiResearch.git", str(PROJECT_DIR)],
            check=True,
        )
else:
    here = Path.cwd().resolve()
    PROJECT_DIR = next(
        (p for p in [here, *here.parents]
         if (p / "train.py").is_file() and (p / "train_deep_rbf.py").is_file()),
        None,
    )
    if PROJECT_DIR is None:
        raise FileNotFoundError(
            "Open this notebook from the refactored ParkinsonAiResearch repository."
        )

# Optional: point this to the extracted refactor package if the code has not
# yet been patched in this runtime.
REFACTOR_DIR = Path(os.environ.get("DEEP_RBF_REFACTOR_DIR", "")).expanduser()
if REFACTOR_DIR and (REFACTOR_DIR / "apply_refactor.py").is_file():
    subprocess.run(
        [sys.executable, str(REFACTOR_DIR / "apply_refactor.py"), "--repo-root", str(PROJECT_DIR)],
        check=True,
    )

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

# Refuse to continue if the obsolete RBF head is still present.
import model.rbf as rbf_module
legacy_rbf_names = [
    name for name in vars(rbf_module)
    if name.lower().startswith("centre") and name.lower().endswith("rbf")
]
if not hasattr(rbf_module, "DeepRBF") or legacy_rbf_names:
    raise RuntimeError(
        "The DeepRBF-only refactor has not been applied in this runtime. Extract the "
        "refactor ZIP and run apply_refactor.py against PROJECT_DIR first."
    )

candidates = []
if os.environ.get("PARKINSON_DATA_ROOT"):
    candidates.append(Path(os.environ["PARKINSON_DATA_ROOT"]).expanduser())
if IN_COLAB:
    drive_project = Path("/content/drive/MyDrive/ParkinsonResearch")
    candidates += [
        drive_project / "ParkinsonData",
        drive_project / "dataset",
        drive_project,
    ]
candidates.append(PROJECT_DIR)

DATA_ROOT = next(
    (p.resolve() for p in candidates
     if (p / "datasets/annotations/fold_manifest.csv").is_file()),
    None,
)
if DATA_ROOT is None:
    raise FileNotFoundError(
        "Could not find datasets/annotations/fold_manifest.csv. "
        "Set PARKINSON_DATA_ROOT to the dataset root and rerun this cell."
    )

MANIFEST = DATA_ROOT / "datasets/annotations/fold_manifest.csv"

# Install only the repository's declared dependencies. Existing compatible
# PyTorch/CUDA packages are retained by pip.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True,
)

import torch
if torch.cuda.is_available():
    TORCH_DEVICE = torch.device("cuda")
    DEVICE = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    TORCH_DEVICE = torch.device("mps")
    DEVICE = "auto"
else:
    TORCH_DEVICE = torch.device("cpu")
    DEVICE = "auto"
print("Project:", PROJECT_DIR)
print("Data root:", DATA_ROOT)
print("Manifest:", MANIFEST)
print("PyTorch:", torch.__version__)
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else DEVICE)

## 1. Configuration

The default is a one-fold pilot. After it succeeds, set `FOLD_END = 25` and rerun the training cells with the same output paths to continue the full 25-fold development schedule.

The augmentation, sampling and numerical hyperparameters below are explicit project defaults. Confirm them against the approved thesis protocol before treating them as exact thesis settings.

In [ ]:
# Pilot first. Change to 25 only after fold 1 has completed successfully.
FOLD_START = 1
FOLD_END = 1

SEED = 42
if torch.cuda.is_available():
    TORCH_DEVICE = torch.device("cuda")
    DEVICE = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    TORCH_DEVICE = torch.device("mps")
    DEVICE = "auto"
else:
    TORCH_DEVICE = torch.device("cpu")
    DEVICE = "auto"

# Stage 1: WideResNet50-2 + Focal Loss
BASELINE_EPOCHS = 15
BASELINE_BATCH_SIZE = 8
BASELINE_LR = 1e-3
BACKBONE_LR = 1e-5
WEIGHT_DECAY = 1e-4
FOCAL_GAMMA = 2.0
FOCAL_ALPHA_MODE = "uniform"
AUGMENTATION = "standard"
SAMPLING = "class_balanced"
ROTATION_DEGREES = 10.0
HFLIP_PROB = 0.5
NOISE_STD = 0.01
CHECKPOINT_EVERY = 5

# Stage 2: DeepRBF
RBF_EPOCHS = 30
RBF_DIM = 64
RBF_DISTANCE = "l1"       # use a separate experiment/output folder for L2
RBF_DISTANCE_POWER = "norm"
KNOWN_MARGIN = 1.0
REJECT_MARGIN = 1.0
BETA = 1.0
RBF_LR = 1e-3
RBF_WEIGHT_DECAY = 0.0
FEATURE_BATCH_SIZE = 8

if IN_COLAB:
    OUTPUT_BASE = Path("/content/drive/MyDrive/ParkinsonResearch/runs/DeepRBF_thesis_v1")
else:
    OUTPUT_BASE = PROJECT_DIR / "outputs" / "DeepRBF_thesis_v1"

BASELINE_DIR = OUTPUT_BASE / "baseline" / "checkpoints"
RBF_ROOT = OUTPUT_BASE / "deep_rbf"
BASELINE_DIR.mkdir(parents=True, exist_ok=True)
RBF_ROOT.mkdir(parents=True, exist_ok=True)

print("Folds:", FOLD_START, "to", FOLD_END)
print("Baseline checkpoints:", BASELINE_DIR)
print("DeepRBF outputs:", RBF_ROOT)

## 2. Stage 1 — train WideResNet50-2 with Focal Loss

This runs the repository's tested `train_all_folds.py` wrapper. Each fold receives its own checkpoint. The classifier is trained with three expert classes (`control`, `parkinson`, `unknown`). The final checkpoint is saved as `*_last.pt`.

The RBF stage will later load the exact matching Focal checkpoint; it will not use a CE checkpoint.

In [ ]:
baseline_command = [
    sys.executable, "-u", "train_all_folds.py",
    "--fold-start", str(FOLD_START),
    "--fold-end", str(FOLD_END),
    "--loss", "focal",
    "--focal-gamma", str(FOCAL_GAMMA),
    "--focal-alpha-mode", FOCAL_ALPHA_MODE,
    "--augmentation", AUGMENTATION,
    "--rotation-degrees", str(ROTATION_DEGREES),
    "--hflip-prob", str(HFLIP_PROB),
    "--noise-std", str(NOISE_STD),
    "--sampling", SAMPLING,
    "--label-source", "consensus",
    "--pretrained",
    "--freeze-policy", "early",
    "--epochs", str(BASELINE_EPOCHS),
    "--batch-size", str(BASELINE_BATCH_SIZE),
    "--image-size", "224",
    "--num-workers", "0",
    "--lr", str(BASELINE_LR),
    "--backbone-lr", str(BACKBONE_LR),
    "--weight-decay", str(WEIGHT_DECAY),
    "--seed", str(SEED),
    "--device", DEVICE,
    "--manifest", str(MANIFEST),
    "--root", str(DATA_ROOT),
    "--checkpoint-dir", str(BASELINE_DIR),
    "--checkpoint-every", str(CHECKPOINT_EVERY),
    "--no-progress",
    "--progress-json",
    "--resume",
]
print(" ".join(baseline_command))
subprocess.run(baseline_command, cwd=PROJECT_DIR, check=True)

## 3. Stage 2 — train DeepRBF on frozen Focal features

The DeepRBF trainer checks that every source checkpoint is:
- WideResNet50-2 with a three-class consensus label head;
- trained with Focal Loss;
- from the same fold manifest and requested fold;
- configured to expose 2048-D pooled features.

Only the DeepRBF affine parameters are optimized in this stage. The WideResNet is frozen and used in evaluation mode. The Focal Loss is not added again to the RBF margin objective.

In [ ]:
rbf_command = [
    sys.executable, "-u", "train_deep_rbf_all_folds.py",
    "--baseline-dir", str(BASELINE_DIR),
    "--baseline-glob", "*_last.pt",
    "--manifest", str(MANIFEST),
    "--root", str(DATA_ROOT),
    "--output-root", str(RBF_ROOT),
    "--fold-start", str(FOLD_START),
    "--fold-end", str(FOLD_END),
    "--epochs", str(RBF_EPOCHS),
    "--rbf-dim", str(RBF_DIM),
    "--distance", RBF_DISTANCE,
    "--distance-power", RBF_DISTANCE_POWER,
    "--known-margin", str(KNOWN_MARGIN),
    "--reject-margin", str(REJECT_MARGIN),
    "--beta", str(BETA),
    "--balance-groups",
    "--lr", str(RBF_LR),
    "--weight-decay", str(RBF_WEIGHT_DECAY),
    "--seed", str(SEED),
    "--device", DEVICE,
    "--feature-batch-size", str(FEATURE_BATCH_SIZE),
    "--resume",
]
print(" ".join(rbf_command))
subprocess.run(rbf_command, cwd=PROJECT_DIR, check=True)

## 4. Stage 3A — select a shared rejection rule from pooled validation

Do not tune a threshold on fold 22: it intentionally has no validation images. The script below selects a fold-local threshold when that fold has a feasible validation candidate. It uses a pooled out-of-fold development fallback for empty fold 22 or any fold without a feasible local candidate. It also checks that a participant is not assigned to validation in multiple folds and writes checkpoint-bound decision files.

No held-out test images are loaded.

In [ ]:
threshold_command = [
    sys.executable, "-u", "tune_deep_rbf_thresholds_all_folds.py",
    "--baseline-dir", str(BASELINE_DIR),
    "--baseline-glob", "*_last.pt",
    "--deep-rbf-root", str(RBF_ROOT),
    "--manifest", str(MANIFEST),
    "--root", str(DATA_ROOT),
    "--fold-start", str(FOLD_START),
    "--fold-end", str(FOLD_END),
    "--distance", RBF_DISTANCE,
    "--distance-power", RBF_DISTANCE_POWER,
    "--selection-unit", "participants",
    "--max-pd-to-control-rate", "0.0",
    "--min-coverage", "0.0",
    "--feature-batch-size", str(FEATURE_BATCH_SIZE),
    "--device", DEVICE,
]
threshold_report_path = RBF_ROOT / "threshold_selection_global" / "threshold_search_report.json"
if threshold_report_path.exists():
    threshold_report_path.unlink()
print(" ".join(threshold_command))
threshold_run = subprocess.run(threshold_command, cwd=PROJECT_DIR, check=False)

if not threshold_report_path.is_file():
    raise RuntimeError(
        f"Threshold tuner failed before writing its report (exit={threshold_run.returncode})."
    )
threshold_report = json.loads(threshold_report_path.read_text(encoding="utf-8"))
THRESHOLDS_READY = threshold_report.get("status") == "selected"
if not THRESHOLDS_READY:
    print(
        "No feasible threshold was found for this pilot. The trained WideResNet and "
        "DeepRBF weights are preserved. Continue with more development folds, then "
        "rerun this cell; no provisional threshold will be treated as a final decision."
    )
else:
    print("Threshold selection complete. Fold-specific decision files are ready.")

## 5. Stage 3B — export combined bundles and evaluate validation (when thresholds are feasible)

Each bundle contains:
- the trained Focal WideResNet weights and preprocessing metadata;
- the separately trained DeepRBF weights and configuration;
- the matching validation-selected rejection thresholds;
- provenance hashes tying the stages together.

The combined model uses `WideResNet.forward_features(images) → DeepRBF(features)`. The baseline classifier logits are not averaged with RBF distances.

In [ ]:
if THRESHOLDS_READY:
    for fold in range(FOLD_START, FOLD_END + 1):
        fold_name = f"fold{fold:02d}_{RBF_DISTANCE}_{RBF_DISTANCE_POWER}"
        fold_dir = RBF_ROOT / fold_name
        rbf_checkpoint = fold_dir / "last.pt"
        decision_config = fold_dir / "threshold_selection" / "rbf_decision.json"
        if not rbf_checkpoint.is_file():
            raise FileNotFoundError(rbf_checkpoint)
        if not decision_config.is_file():
            raise FileNotFoundError(decision_config)

        bundle_path = fold_dir / "combined_model.pt"
        export_command = [
            sys.executable, "export_deep_rbf_bundle.py",
            "--deep-rbf-checkpoint", str(rbf_checkpoint),
            "--decision-config", str(decision_config),
            "--manifest", str(MANIFEST),
            "--root", str(DATA_ROOT),
            "--output", str(bundle_path),
            "--device", DEVICE,
        ]
        subprocess.run(export_command, cwd=PROJECT_DIR, check=True)

        validation_command = [
            sys.executable, "eval_deep_rbf.py",
            "--checkpoint", str(rbf_checkpoint),
            "--manifest", str(MANIFEST),
            "--root", str(DATA_ROOT),
            "--split", "val",
            "--decision-config", str(decision_config),
            "--feature-batch-size", str(FEATURE_BATCH_SIZE),
            "--device", DEVICE,
            "--output-dir", str(fold_dir / "combined_validation"),
        ]
        subprocess.run(validation_command, cwd=PROJECT_DIR, check=True)

    print("Combined bundles and validation reports are saved under:", RBF_ROOT)
else:
    print("Bundle export/evaluation is deferred until a feasible development threshold exists.")

## 6. Compose the two trained models in Python

This cell demonstrates the actual composed forward pass on one nonempty development validation fold. It reports distances and decisions; it does not use expert or clinical labels as model inputs.

For a full 25-fold run, fold 1 is selected for this demonstration. If you run only fold 22, choose another fold with nonempty validation.

In [ ]:
from model.deep_rbf_system import load_deep_rbf_bundle, WideResNetDeepRBF
from utils.rbf_pipeline import load_deep_rbf_checkpoint
from config import PreprocessingConfig
from data.dataloader import SpiralDataset, build_eval_transform
from torch.utils.data import DataLoader

demo_fold = next((f for f in range(FOLD_START, FOLD_END + 1) if f != 22), None)
if demo_fold is None:
    raise ValueError("The selected fold range contains no nonempty validation fold.")

demo_dir = RBF_ROOT / f"fold{demo_fold:02d}_{RBF_DISTANCE}_{RBF_DISTANCE_POWER}"
if THRESHOLDS_READY:
    system, bundle = load_deep_rbf_bundle(
        demo_dir / "combined_model.pt", device=TORCH_DEVICE
    )
    decision = bundle["decision"]
    preprocessing_data = bundle["baseline"]["preprocessing"]
else:
    # Forward-only composition for a pilot whose validation set is too small
    # to select a defensible rejection threshold.
    backbone, rbf_head, rbf_checkpoint = load_deep_rbf_checkpoint(
        demo_dir / "last.pt", MANIFEST, DATA_ROOT, TORCH_DEVICE
    )
    system = WideResNetDeepRBF(backbone, rbf_head).to(TORCH_DEVICE).eval()
    preprocessing_data = rbf_checkpoint["source_baseline"]["preprocessing"]
    decision = None

preprocessing = PreprocessingConfig.from_dict(preprocessing_data["config"])
val_dataset = SpiralDataset(
    manifest_path=MANIFEST,
    root=DATA_ROOT,
    fold_id=demo_fold,
    split="val",
    image_size=preprocessing.image_size,
    label_source="consensus",
    transform=build_eval_transform(preprocessing),
    allow_empty=True,
)
if len(val_dataset) == 0:
    print(f"Fold {demo_fold} has empty validation; select another fold for the demo.")
else:
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0)
    batch = next(iter(val_loader))
    with torch.no_grad():
        distances = system(batch["image"].to(TORCH_DEVICE))
    if decision is not None:
        labels, reasons, distances = system.predict(
            batch["image"].to(TORCH_DEVICE),
            distance_threshold=float(decision["distance_threshold"]),
            gap_threshold=float(decision["gap_threshold"]),
        )
        names = {0: "control", 1: "parkinson", 2: "unknown"}
        for i, drawing_id in enumerate(batch["drawing_id"]):
            print({
                "drawing_id": drawing_id,
                "distance_control": float(distances[i, 0].cpu()),
                "distance_parkinson": float(distances[i, 1].cpu()),
                "prediction": names[int(labels[i].cpu())],
                "reason": reasons[i],
            })
    else:
        print("Combined forward pass succeeded. No threshold was selected on this small pilot.")
        for i, drawing_id in enumerate(batch["drawing_id"]):
            print({
                "drawing_id": drawing_id,
                "distance_control": float(distances[i, 0].cpu()),
                "distance_parkinson": float(distances[i, 1].cpu()),
            })

## 7. Final checks before any held-out test evaluation

- Confirm that all requested Focal baseline checkpoints exist and their `loss.name` is `focal`.
- Confirm that every fold has a `deep_rbf/.../last.pt`.
- Confirm the global threshold report lists the expected nonempty validation folds; fold 22 should be skipped for threshold fitting.
- Review the selected radius/gap and participant-level validation report.
- Freeze the full protocol before opening the held-out test partition.

The notebook deliberately stops at development validation. Do not select hyperparameters or thresholds from the test partition.